# Etapa 3 · Punto 1 — Carga del grafo de conocimiento en Neo4j

**Objetivo.** Cargar en Neo4j (AuraDB) la ontología y las instancias generadas en la Etapa 2 y verificar,
mediante consultas Cypher, que las principales entidades y relaciones se recuperan correctamente antes de
construir el sistema de recomendación.

Se sigue el mismo mecanismo del taller *"bases de datos en grafos y recomendación de artículos científicos"*:

1. Leer el RDF con `rdflib`.
2. Extraer filas (diccionarios) **desde los triples RDF**.
3. Importarlas a Neo4j por lotes con `UNWIND` + `MERGE` usando el driver oficial de Python
   (`MERGE` hace la carga idempotente: se puede re-ejecutar sin duplicar nodos).
4. Verificar con consultas Cypher.

**Archivos de entrada** (carpeta `entrega_3/data/`):

| Archivo | Contenido |
|---|---|
| `ontology.ttl` | Ontología RDFS/OWL de la Etapa 2 (clases, propiedades, dominios/rangos, restricciones OWL y clases del bonus). |
| `instances.ttl` | Instancias: canciones (`Track`), artistas, países, tags y similitudes Last.fm. |
| `enrichment.ttl` | *(Opcional)* Enriquecimiento del bonus de la Etapa 2: géneros, instrumentos y personas extraídos de textos, con evidencia. |

## 0. Preparar el entorno

Funciona tanto en local (Jupyter / VS Code, desde `entrega_3/notebooks/`) como en Google Colab.
En Colab se pedirá subir los archivos `.ttl`.

In [9]:
%pip -q install "neo4j>=5.26,<7" "rdflib>=7,<8" "pandas>=2"


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os, re, sys, time
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
pd.set_option('display.max_colwidth', 120)

EN_COLAB = 'google.colab' in sys.modules
DATA_DIR = Path(os.environ.get('DATA_DIR', '../data'))

if EN_COLAB:
    from google.colab import files
    DATA_DIR = Path('/content/data'); DATA_DIR.mkdir(exist_ok=True)
    print('Sube ontology.ttl, instances.ttl, .env y (opcional) enrichment.ttl')
    for nombre, contenido in files.upload().items():
        (DATA_DIR / nombre).write_bytes(contenido)

ARCHIVO_ONTOLOGIA = DATA_DIR / 'ontology.ttl'
ARCHIVO_INSTANCIAS = DATA_DIR / 'instances.ttl'
ARCHIVO_ENRIQUECIMIENTO = DATA_DIR / 'enrichment.ttl'
INCLUIR_ENRIQUECIMIENTO = ARCHIVO_ENRIQUECIMIENTO.exists()

for f in (ARCHIVO_ONTOLOGIA, ARCHIVO_INSTANCIAS, ARCHIVO_ENRIQUECIMIENTO):
    print(f"{'✓' if f.exists() else '✗'} {f}  ({f.stat().st_size/1e6:.1f} MB)" if f.exists() else f'✗ {f}')

✓ ../data/ontology.ttl  (0.0 MB)
✓ ../data/instances.ttl  (40.3 MB)
✓ ../data/enrichment.ttl  (0.0 MB)


## 1. Lectura de la ontología y las instancias con `rdflib`

In [11]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, URIRef, BNode, Literal
from rdflib.collection import Collection

RES = Namespace('http://uniandes.edu.co/wsproject2026/vocab/lit#')       # clases e individuos
RBO = Namespace('http://uniandes.edu.co/wsproject2026/vocab/ontology#')  # propiedades

t0 = time.time()
onto = Graph().parse(ARCHIVO_ONTOLOGIA, format='turtle')
inst = Graph().parse(ARCHIVO_INSTANCIAS, format='turtle')
enr = Graph()
if INCLUIR_ENRIQUECIMIENTO:
    enr.parse(ARCHIVO_ENRIQUECIMIENTO, format='turtle')

print(f'Triples ontología:       {len(onto):>9,}')
print(f'Triples instancias:      {len(inst):>9,}')
print(f'Triples enriquecimiento: {len(enr):>9,}')
print(f'Tiempo de lectura: {time.time() - t0:.1f}s')

Triples ontología:             159
Triples instancias:      1,039,960
Triples enriquecimiento:       490
Tiempo de lectura: 13.8s


In [12]:
def local(uri):
    # Nombre local de una URI: ...#Track_TRAAAFW128F42A4CFD -> Track_TRAAAFW128F42A4CFD
    s = str(uri)
    return s.split('#')[-1] if '#' in s else s.rstrip('/').split('/')[-1]

def tipo_relacion(nombre):
    # Convención Neo4j para tipos de relación: sungBy -> SUNG_BY, hasTag -> HAS_TAG
    return re.sub(r'(?<!^)(?=[A-Z])', '_', nombre).upper()

def texto(g, s, p, lang=None):
    # Primer literal de (s, p) — opcionalmente filtrado por idioma.
    for o in g.objects(s, p):
        if lang is None or getattr(o, 'language', None) == lang:
            return str(o)
    return None

def numero(g, s, p, cast):
    v = g.value(s, p)
    return None if v is None else cast(str(v))

### 1.1 Diagnóstico: ¿qué usan las instancias y qué declara la ontología?

Antes de cargar, se contrasta el vocabulario **usado** en las instancias con el **declarado** en la ontología.
Esto permite decidir el mapeo a Neo4j y detectar inconsistencias heredadas de la Etapa 2.

In [13]:
RDF_SUBCLASS = URIRef(str(RDF) + 'subClassOf')   # uso no estándar presente en la ontología
SUBCLASS_PREDS = (RDFS.subClassOf, RDF_SUBCLASS)
TIPOS_CLASE = (RDFS.Class, OWL.Class)
TIPOS_PROPIEDAD = (RDF.Property, OWL.ObjectProperty, OWL.DatatypeProperty)
VOCAB_BASE = {RDF.type, RDFS.label, RDFS.comment, RDF.subject, RDF.predicate, RDF.object}

clases_declaradas = {s for t in TIPOS_CLASE for s in onto.subjects(RDF.type, t)}
props_declaradas = {s for t in TIPOS_PROPIEDAD for s in onto.subjects(RDF.type, t)}

clases_usadas, preds_usados = Counter(), Counter()
for g in (inst, enr):
    for s, p, o in g:
        preds_usados[p] += 1
        if p == RDF.type:
            clases_usadas[o] += 1

display(pd.DataFrame(
    [{'clase': local(c), 'instancias': n, 'declarada_en_ontologia': c in clases_declaradas}
     for c, n in clases_usadas.most_common()]))
display(pd.DataFrame(
    [{'predicado': local(p), 'triples': n, 'declarado_en_ontologia': p in props_declaradas or p in VOCAB_BASE}
     for p, n in preds_usados.most_common()]))

,clase,instancias,declarada_en_ontologia
0,Track,100000,True
1,Similarity,60962,False
2,Artist,27378,True
3,Tag,245,True
4,Country,138,True
5,ExtractedFact,47,True
6,TextSource,47,True
7,Genre,23,True
8,Instrument,6,True
9,Person,5,True


,predicado,triples,declarado_en_ontologia
0,type,188851,True
1,year,100000,True
2,duration,100000,True
3,released,100000,True
4,sungBy,100000,True
5,title,99997,True
6,similarityScore,60962,False
7,hasSimilarity,60962,False
8,similarTrack,60962,False
9,label,32369,True


In [14]:
# Observaciones automáticas sobre la ontología
referenciadas = set()
for p in (RDFS.domain, RDFS.range, OWL.disjointWith, OWL.onClass, OWL.allValuesFrom, *SUBCLASS_PREDS):
    for o in onto.objects(None, p):
        if isinstance(o, URIRef) and not str(o).startswith(str(XSD)):
            referenciadas.add(o)
for s in onto.subjects(OWL.disjointWith, None):
    referenciadas.add(s)

observaciones = []
n_rdf_sub = len(list(onto.triples((None, RDF_SUBCLASS, None))))
if n_rdf_sub:
    observaciones.append(f'{n_rdf_sub} axiomas usan rdf:subClassOf (no estándar); se interpretan como rdfs:subClassOf.')
for c in sorted(referenciadas - clases_declaradas):
    observaciones.append(f'Clase referenciada pero no declarada: {local(c)}')
for c in sorted(set(clases_usadas) - clases_declaradas):
    observaciones.append(f'Clase usada en instancias pero no declarada: {local(c)}')
for p in sorted(set(preds_usados) - props_declaradas - VOCAB_BASE):
    observaciones.append(f'Predicado usado en instancias pero no declarado: {local(p)}')

tracks_rdf = set(inst.subjects(RDF.type, RES.Track))
sim_total = len(list(inst.objects(None, RBO.similarTrack)))
sim_internas = sum(1 for o in inst.objects(None, RBO.similarTrack) if o in tracks_rdf)
observaciones.append(f'Similitudes Last.fm: {sim_total:,} en total; {sim_internas:,} apuntan a canciones del dataset '
                     f'y {sim_total - sim_internas:,} a canciones externas sin datos.')
anio_cero = sum(1 for o in inst.objects(None, RBO.year) if str(o) == '0')
observaciones.append(f'{anio_cero:,} canciones tienen year = "0" (año desconocido).')

for o in observaciones:
    print('•', o)

• 3 axiomas usan rdf:subClassOf (no estándar); se interpretan como rdfs:subClassOf.
• Clase referenciada pero no declarada: Persona
• Clase usada en instancias pero no declarada: Similarity
• Predicado usado en instancias pero no declarado: hasSimilarity
• Predicado usado en instancias pero no declarado: similarTrack
• Predicado usado en instancias pero no declarado: similarityScore
• Similitudes Last.fm: 60,962 en total; 6,380 apuntan a canciones del dataset y 54,582 a canciones externas sin datos.
• 45,346 canciones tienen year = "0" (año desconocido).


## 2. Diseño del grafo de propiedades (RDF → Neo4j)

Neo4j usa un **modelo de grafo de propiedades** (nodos con etiquetas, relaciones con tipo y propiedades en ambos).
Las decisiones de mapeo son:

| RDF (Etapa 2) | Neo4j | Decisión |
|---|---|---|
| `res:Track` | `(:Track:CreativeWork)` | La jerarquía `Track ⊑ CreativeWork` se materializa como etiqueta adicional. |
| `res:Artist` | `(:Artist:Person)` | `Artist ⊑ Person`. `name` = nombre canónico; `labels` = todos los alias (`rdfs:label`). |
| `res:Country` | `(:Country:GeographicalEntity)` | `Country ⊑ GeographicalEntity`. Las múltiples `rbo:area` se guardan en la lista `areas`. |
| `res:Tag` | `(:Tag)` | `name` se recupera del literal `rbo:lastfm_top_tag` original. |
| Propiedades de datos (`rbo:title`, `rbo:year`, …) | Propiedades del nodo | Se conservan los nombres de la ontología. `year = 0` se omite (desconocido). |
| `rbo:sungBy` | `(:Track)-[:SUNG_BY]->(:Artist)` | Relación directa. |
| `rbo:hasTag` | `(:Track)-[:HAS_TAG]->(:Tag)` | Relación directa. |
| `rbo:origin` | `(:Artist)-[:ORIGIN]->(:Country)` | Relación directa. |
| `rbo:hasSimilarity` + nodo en blanco `res:Similarity` (`similarTrack`, `similarityScore`) | `(:Track)-[:SIMILAR_TO {score}]->(:Track)` | La reificación con nodo en blanco se colapsa en **una relación con propiedad**, que es el idioma natural del grafo de propiedades. Solo se cargan similitudes entre canciones del dataset. |
| Bonus: `rbo:associatedWithGenre`, `rbo:playsInstrument`, … | `(:Artist)-[:ASSOCIATED_WITH_GENRE {evidence, sources}]->(:Genre)`, … | Los `res:ExtractedFact` (reificación de la evidencia) se guardan como propiedades de la relación. |
| Ontología (clases, propiedades, dominios, rangos, restricciones OWL) | Meta-grafo `(:OntologyClass)`, `(:OntologyProperty)`, `(:Datatype)`, `(:Restriction)` | La ontología queda consultable en Neo4j junto a los datos. |

Todas las entidades conservan su **URI** original (`uri`, con restricción de unicidad) y un identificador corto (`id`).

In [15]:
# Relaciones entre instancias: (predicado RDF, grafo de origen). El tipo de relación se deriva del nombre
# del predicado y las etiquetas de origen/destino del dominio/rango declarados en la ontología.
RELACIONES_DIRECTAS = [
    (RBO.sungBy, inst),
    (RBO.hasTag, inst),
    (RBO.origin, inst),
    (RBO.associatedWithGenre, enr),
    (RBO.influencedByGenre, enr),
    (RBO.playsInstrument, enr),
    (RBO.influencedByArtist, enr),
    (RBO.collaboratedWith, enr),
    (RBO.producedBy, enr),
]

# Propiedades de datos de cada clase -> nombre de la propiedad en Neo4j
PROPIEDADES_TRACK = {
    RBO.title: 'title', RBO.released: 'released', RBO.duration: 'duration',
    RBO.lastfm_top_tag: 'lastfm_top_tag', RBO.lastfm_top_tag_score: 'lastfm_top_tag_score',
    RBO.key: 'key', RBO.keyScale: 'keyScale', RBO.tuningFrequency: 'tuningFrequency', RBO.year: 'year',
}
PROPIEDADES_ARTIST = {RDFS.label: 'name', RBO.gender: 'gender', RBO.artist_type: 'artist_type'}

def superclases(clase):
    # Clase + todas sus superclases nombradas (cierre transitivo de subClassOf).
    vistas, pendientes = [], [clase]
    while pendientes:
        c = pendientes.pop()
        if c in vistas:
            continue
        vistas.append(c)
        for p in SUBCLASS_PREDS:
            pendientes += [o for o in onto.objects(c, p) if isinstance(o, URIRef)]
    return [local(c) for c in vistas]

def dominio_rango(pred):
    return local(onto.value(pred, RDFS.domain)), local(onto.value(pred, RDFS.range))

for clase in (RES.Track, RES.Artist, RES.Country, RES.Tag, RES.Genre, RES.Instrument, RES.Person):
    print(f'{local(clase):<11} -> etiquetas Neo4j {superclases(clase)}')
for pred, _ in RELACIONES_DIRECTAS:
    d, r = dominio_rango(pred)
    print(f'(:{d})-[:{tipo_relacion(local(pred))}]->(:{r})')

Track       -> etiquetas Neo4j ['Track', 'CreativeWork']
Artist      -> etiquetas Neo4j ['Artist', 'Person']
Country     -> etiquetas Neo4j ['Country', 'GeographicalEntity']
Tag         -> etiquetas Neo4j ['Tag']
Genre       -> etiquetas Neo4j ['Genre']
Instrument  -> etiquetas Neo4j ['Instrument']
Person      -> etiquetas Neo4j ['Person']
(:Track)-[:SUNG_BY]->(:Artist)
(:Track)-[:HAS_TAG]->(:Tag)
(:Artist)-[:ORIGIN]->(:Country)
(:Artist)-[:ASSOCIATED_WITH_GENRE]->(:Genre)
(:Artist)-[:INFLUENCED_BY_GENRE]->(:Genre)
(:Artist)-[:PLAYS_INSTRUMENT]->(:Instrument)
(:Artist)-[:INFLUENCED_BY_ARTIST]->(:Person)
(:Artist)-[:COLLABORATED_WITH]->(:Person)
(:Artist)-[:PRODUCED_BY]->(:Person)


## 3. Conexión segura a Neo4j AuraDB

Las credenciales **no se guardan en el notebook**: se leen del archivo `entrega_3/.env`
(plantilla en `.env.example`, excluido de git). En Colab, subir también el `.env` junto a los `.ttl`.
URI esperada: `neo4j+s://<id>.databases.neo4j.io`.

In [16]:
from neo4j import GraphDatabase

def cargar_env(ruta):
    # Lector mínimo de archivos .env (CLAVE=valor), sin dependencias extra.
    for linea in ruta.read_text(encoding='utf-8').splitlines():
        if '=' in linea and not linea.lstrip().startswith('#'):
            k, v = linea.split('=', 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")

# entrega_3/.env (ejecutando desde notebooks/), ./.env, o el .env subido en Colab
CANDIDATOS_ENV = [Path('../.env'), Path('.env'), DATA_DIR / '.env', Path('entrega_3/.env')]
ARCHIVO_ENV = next((p for p in CANDIDATOS_ENV if p.exists()), None)
if ARCHIVO_ENV is None:
    raise FileNotFoundError('No se encontró entrega_3/.env. Cópialo desde .env.example y completa las credenciales.')
cargar_env(ARCHIVO_ENV)

faltantes = [k for k in ('NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD') if not os.environ.get(k)]
if faltantes:
    raise ValueError(f'Faltan variables en {ARCHIVO_ENV.resolve()}: {faltantes}')

NEO4J_URI = os.environ['NEO4J_URI']
NEO4J_USER = os.environ['NEO4J_USER']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.environ.get('NEO4J_DATABASE') or None   # None = base de datos por defecto
print(f'Credenciales leídas de {ARCHIVO_ENV.resolve()} (usuario: {NEO4J_USER})')

from neo4j.exceptions import DriverError, ServiceUnavailable, SessionExpired

def conectar(intentos=5, espera=10):
    # AuraDB Free puede rechazar conexiones de forma intermitente: se reintenta antes de fallar.
    for i in range(1, intentos + 1):
        d = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        try:
            d.verify_connectivity()
            return d
        except (ServiceUnavailable, SessionExpired) as e:
            d.close()
            if i == intentos:
                raise
            print(f'Aura no disponible ({type(e).__name__}); reintento {i}/{intentos - 1} en {espera}s...')
            time.sleep(espera)

def sesion():
    # Abre una sesión; si el driver fue cerrado (p. ej. tras ejecutar la última celda), se reconecta.
    global driver
    try:
        return driver.session(database=NEO4J_DATABASE)
    except DriverError:
        driver = conectar()
        return driver.session(database=NEO4J_DATABASE)

driver = conectar()
print('Conexión establecida con', driver.get_server_info().agent)

Credenciales leídas de /Users/gabrielgomezcorredor/Desktop/Gabriel_Gomez/Maestria MISIS/Semantic Web/semantic_web_project/entrega_3/.env (usuario: 218cfcc4)
Conexión establecida con Neo4j/5.27-aura


In [17]:
BATCH_SIZE = 1000

def run_cypher(query, **parameters):
    with sesion() as session:
        return [dict(r) for r in session.run(query, **parameters)]

def cargar_en_lotes(nombre, query, rows, batch_size=BATCH_SIZE):
    # UNWIND por lotes dentro de transacciones gestionadas (reintentos automáticos ante errores transitorios).
    t = time.time()
    with sesion() as session:
        for i in range(0, len(rows), batch_size):
            session.execute_write(lambda tx, lote: tx.run(query, rows=lote).consume(), rows[i:i + batch_size])
    print(f'✓ {nombre:<40} {len(rows):>8,}  ({time.time() - t:.1f}s)')

### 3.1 (Opcional) Reiniciar los datos del proyecto

La carga es idempotente (`MERGE`), así que no es necesario borrar para re-ejecutar. Si se quiere empezar de cero,
poner `REINICIAR = True`: solo se borran los nodos con etiquetas de **este** proyecto (no los del taller, p. ej. `Paper`).

In [18]:
REINICIAR = False

ETIQUETAS_PROYECTO = ['Track', 'Artist', 'Person', 'Country', 'Tag', 'Genre', 'Instrument',
                      'CreativeWork', 'GeographicalEntity',
                      'OntologyClass', 'OntologyProperty', 'Datatype', 'Restriction']
if REINICIAR:
    total = 0
    while True:
        borrados = run_cypher('''
            MATCH (n) WHERE any(l IN labels(n) WHERE l IN $labels)
            WITH n LIMIT 10000 DETACH DELETE n RETURN count(*) AS borrados''',
            labels=ETIQUETAS_PROYECTO)[0]['borrados']
        total += borrados
        if borrados == 0:
            break
    print('Nodos eliminados:', total)

## 4. Restricciones de unicidad

Una restricción por etiqueta sobre `uri` evita duplicados y crea el índice que usan los `MERGE`/`MATCH` de la carga.

In [19]:
ETIQUETAS_CON_URI = ['Track', 'Artist', 'Person', 'Country', 'Tag', 'Genre', 'Instrument',
                     'OntologyClass', 'OntologyProperty', 'Datatype']
for etiqueta in ETIQUETAS_CON_URI:
    run_cypher(f'CREATE CONSTRAINT {etiqueta.lower()}_uri_unique IF NOT EXISTS '
               f'FOR (n:{etiqueta}) REQUIRE n.uri IS UNIQUE')
run_cypher('CREATE CONSTRAINT restriction_id_unique IF NOT EXISTS FOR (n:Restriction) REQUIRE n.id IS UNIQUE')
run_cypher('CREATE INDEX track_id IF NOT EXISTS FOR (n:Track) ON (n.id)')
run_cypher('CREATE INDEX artist_name IF NOT EXISTS FOR (n:Artist) ON (n.name)')

display(pd.DataFrame(run_cypher('SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties RETURN name, labelsOrTypes, properties ORDER BY name')))

,name,labelsOrTypes,properties
0,artist_uri_unique,[Artist],[uri]
1,country_uri_unique,[Country],[uri]
2,datatype_uri_unique,[Datatype],[uri]
3,genre_uri_unique,[Genre],[uri]
4,instrument_uri_unique,[Instrument],[uri]
5,ontologyclass_uri_unique,[OntologyClass],[uri]
6,ontologyproperty_uri_unique,[OntologyProperty],[uri]
7,person_uri_unique,[Person],[uri]
8,restriction_id_unique,[Restriction],[id]
9,tag_uri_unique,[Tag],[uri]


## 5. Importación de la ontología → meta-grafo

Se representa la ontología como un grafo (inspirado en la importación de ontologías de *neosemantics*):

- `(:OntologyClass)-[:SUBCLASS_OF]->(:OntologyClass)` y `[:DISJOINT_WITH]`
- `(:OntologyProperty)-[:DOMAIN]->(:OntologyClass)`, `[:RANGE]->(:OntologyClass | :Datatype)`
- `(:OntologyProperty)-[:SUBPROPERTY_OF | INVERSE_OF]->(:OntologyProperty)`, `[:CHAIN_STEP {order}]` (cadena de propiedades)
- `(:OntologyClass)-[:SUBCLASS_OF]->(:Restriction {kind, value})-[:ON_PROPERTY]->(:OntologyProperty)`, `-[:ON_CLASS]->(:OntologyClass)`

Cada propiedad indica además cómo quedó representada en Neo4j (`neo4j_mapping`), lo que conecta el esquema con los datos.

In [20]:
es_xsd = lambda u: str(u).startswith(str(XSD))

# --- Clases (declaradas + referenciadas) ---
todas_clases = clases_declaradas | referenciadas
class_rows = [{
    'uri': str(c), 'name': local(c), 'declared': c in clases_declaradas,
    'label_es': texto(onto, c, RDFS.label, 'es'), 'label_en': texto(onto, c, RDFS.label, 'en'),
    'comment_es': texto(onto, c, RDFS.comment, 'es'), 'comment_en': texto(onto, c, RDFS.comment, 'en'),
} for c in todas_clases]

# --- Propiedades (declaradas + referenciadas en inverseOf / subPropertyOf / restricciones / cadenas) ---
todas_props = set(props_declaradas)
for s, o in onto.subject_objects(OWL.inverseOf):
    todas_props |= {s, o}
todas_props |= set(onto.objects(None, RDFS.subPropertyOf)) | set(onto.objects(None, OWL.onProperty))
cadenas = {p: [x for x in Collection(onto, lista)] for p, lista in onto.subject_objects(OWL.propertyChainAxiom)}
for p, pasos in cadenas.items():
    todas_props |= {p, *pasos}

MAPEO_NEO4J = {p: f'(:{dominio_rango(p)[0]})-[:{tipo_relacion(local(p))}]->(:{dominio_rango(p)[1]})'
               for p, _ in RELACIONES_DIRECTAS}
MAPEO_NEO4J.update({p: f'Track.{n}' for p, n in PROPIEDADES_TRACK.items()})
MAPEO_NEO4J.update({RBO.gender: 'Artist.gender', RBO.artist_type: 'Artist.artist_type', RBO.area: 'Country.areas'})

def tipo_propiedad(p):
    rango = onto.value(p, RDFS.range)
    if (p, RDF.type, OWL.DatatypeProperty) in onto or (rango is not None and es_xsd(rango)):
        return 'DatatypeProperty'
    if (p, RDF.type, OWL.ObjectProperty) in onto or rango is not None:
        return 'ObjectProperty'
    return 'Property'

prop_rows = [{
    'uri': str(p), 'name': local(p), 'declared': p in props_declaradas, 'kind': tipo_propiedad(p),
    'neo4j_mapping': MAPEO_NEO4J.get(p),
    'chain': [local(x) for x in cadenas.get(p, [])] or None,
} for p in todas_props]

datatype_rows = [{'uri': str(o), 'name': 'xsd:' + local(o)} for o in set(onto.objects(None, RDFS.range)) if es_xsd(o)]

# --- Aristas del meta-grafo ---
par = lambda s, o: {'s': str(s), 'o': str(o)}
subclass_rows = [dict(par(s, o), via='rdfs:subClassOf' if p == RDFS.subClassOf else 'rdf:subClassOf')
                 for p in SUBCLASS_PREDS for s, o in onto.subject_objects(p) if isinstance(o, URIRef)]
domain_rows = [par(s, o) for s, o in onto.subject_objects(RDFS.domain)]
range_class_rows = [par(s, o) for s, o in onto.subject_objects(RDFS.range) if not es_xsd(o)]
range_dt_rows = [par(s, o) for s, o in onto.subject_objects(RDFS.range) if es_xsd(o)]
subprop_rows = [par(s, o) for s, o in onto.subject_objects(RDFS.subPropertyOf)]
inverse_rows = [par(s, o) for s, o in onto.subject_objects(OWL.inverseOf)]
disjoint_rows = [par(s, o) for s, o in onto.subject_objects(OWL.disjointWith)]
chain_rows = [{'s': str(p), 'o': str(x), 'order': i + 1} for p, pasos in cadenas.items() for i, x in enumerate(pasos)]

# --- Restricciones OWL (nodos en blanco owl:Restriction) ---
TIPOS_RESTRICCION = [OWL.minQualifiedCardinality, OWL.maxQualifiedCardinality, OWL.qualifiedCardinality,
                     OWL.cardinality, OWL.minCardinality, OWL.maxCardinality, OWL.allValuesFrom, OWL.someValuesFrom]
restriction_rows = []
for p in SUBCLASS_PREDS:
    for clase, r in onto.subject_objects(p):
        if not isinstance(r, BNode) or (r, RDF.type, OWL.Restriction) not in onto:
            continue
        prop = onto.value(r, OWL.onProperty)
        for tr in TIPOS_RESTRICCION:
            valor = onto.value(r, tr)
            if valor is None:
                continue
            clase_obj = onto.value(r, OWL.onClass) or (valor if isinstance(valor, URIRef) else None)
            restriction_rows.append({
                'id': f'{local(clase)}|{local(prop)}|{local(tr)}|{local(valor)}',
                'class': str(clase), 'property': str(prop), 'kind': local(tr),
                'value': local(valor) if isinstance(valor, URIRef) else int(str(valor)),
                'on_class': str(clase_obj) if clase_obj is not None else None,
            })

print(f'Clases: {len(class_rows)} | Propiedades: {len(prop_rows)} | Tipos de dato: {len(datatype_rows)} | '
      f'Restricciones OWL: {len(restriction_rows)}')

Clases: 12 | Propiedades: 28 | Tipos de dato: 4 | Restricciones OWL: 4


In [21]:
ONTO_STEPS = [
    ('Ontología: clases', 'UNWIND $rows AS row MERGE (n:OntologyClass {uri: row.uri}) SET n += row', class_rows),
    ('Ontología: propiedades', 'UNWIND $rows AS row MERGE (n:OntologyProperty {uri: row.uri}) SET n += row', prop_rows),
    ('Ontología: tipos de dato', 'UNWIND $rows AS row MERGE (n:Datatype {uri: row.uri}) SET n += row', datatype_rows),
    ('Ontología: SUBCLASS_OF', '''UNWIND $rows AS row MATCH (a:OntologyClass {uri: row.s}), (b:OntologyClass {uri: row.o})
        MERGE (a)-[r:SUBCLASS_OF]->(b) SET r.via = row.via''', subclass_rows),
    ('Ontología: DOMAIN', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyClass {uri: row.o})
        MERGE (a)-[:DOMAIN]->(b)''', domain_rows),
    ('Ontología: RANGE (clases)', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyClass {uri: row.o})
        MERGE (a)-[:RANGE]->(b)''', range_class_rows),
    ('Ontología: RANGE (tipos de dato)', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:Datatype {uri: row.o})
        MERGE (a)-[:RANGE]->(b)''', range_dt_rows),
    ('Ontología: SUBPROPERTY_OF', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyProperty {uri: row.o})
        MERGE (a)-[:SUBPROPERTY_OF]->(b)''', subprop_rows),
    ('Ontología: INVERSE_OF', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyProperty {uri: row.o})
        MERGE (a)-[:INVERSE_OF]->(b)''', inverse_rows),
    ('Ontología: DISJOINT_WITH', '''UNWIND $rows AS row MATCH (a:OntologyClass {uri: row.s}), (b:OntologyClass {uri: row.o})
        MERGE (a)-[:DISJOINT_WITH]->(b)''', disjoint_rows),
    ('Ontología: CHAIN_STEP', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyProperty {uri: row.o})
        MERGE (a)-[r:CHAIN_STEP {order: row.order}]->(b)''', chain_rows),
    ('Ontología: restricciones OWL', '''UNWIND $rows AS row
        MATCH (c:OntologyClass {uri: row.class}), (p:OntologyProperty {uri: row.property})
        MERGE (r:Restriction {id: row.id}) SET r.kind = row.kind, r.value = row.value
        MERGE (c)-[:SUBCLASS_OF]->(r)
        MERGE (r)-[:ON_PROPERTY]->(p)
        WITH r, row WHERE row.on_class IS NOT NULL
        MATCH (oc:OntologyClass {uri: row.on_class})
        MERGE (r)-[:ON_CLASS]->(oc)''', restriction_rows),
]
for nombre, query, rows in ONTO_STEPS:
    cargar_en_lotes(nombre, query, rows)

✓ Ontología: clases                              12  (0.6s)
✓ Ontología: propiedades                         28  (0.5s)
✓ Ontología: tipos de dato                        4  (0.6s)
✓ Ontología: SUBCLASS_OF                          3  (0.5s)
✓ Ontología: DOMAIN                              25  (0.6s)
✓ Ontología: RANGE (clases)                      12  (0.5s)
✓ Ontología: RANGE (tipos de dato)               14  (0.5s)
✓ Ontología: SUBPROPERTY_OF                       1  (0.5s)
✓ Ontología: INVERSE_OF                           2  (0.5s)
✓ Ontología: DISJOINT_WITH                       15  (0.5s)
✓ Ontología: CHAIN_STEP                           2  (0.6s)
✓ Ontología: restricciones OWL                    4  (0.6s)


## 6. Extracción de filas **desde RDF** (instancias)

Como en el taller, las filas se construyen leyendo los triples RDF (no el CSV original).

In [22]:
t0 = time.time()

def fila_track(s):
    fila = {'uri': str(s), 'id': local(s).removeprefix('Track_')}
    for p, nombre in PROPIEDADES_TRACK.items():
        v = inst.value(s, p)
        if v is None:
            continue
        if p == RBO.year:
            anio = int(str(v))
            fila[nombre] = anio if anio > 0 else None          # year "0" = desconocido
        elif p in (RBO.duration, RBO.lastfm_top_tag_score, RBO.tuningFrequency):
            fila[nombre] = float(str(v))
        else:
            fila[nombre] = str(v)
    return fila

def fila_artist(s):
    # Nombre canónico: el alias más corto, evitando los que traen '_' o ';' (artefactos del CSV, p. ej. 'Ike_ Tina Turner')
    alias = sorted({str(o) for o in inst.objects(s, RDFS.label)},
                   key=lambda x: (any(c in x for c in '_;'), len(x), x))
    return {'uri': str(s), 'id': local(s).removeprefix('Artist_'),
            'name': alias[0] if alias else None, 'labels': alias,
            'gender': texto(inst, s, RBO.gender), 'artist_type': texto(inst, s, RBO.artist_type)}

def fila_country(s):
    areas = sorted({str(o) for o in inst.objects(s, RBO.area)})
    return {'uri': str(s), 'id': local(s).removeprefix('Country_'), 'areas': areas, 'n_areas': len(areas)}

# Nombre original del tag: el literal rbo:lastfm_top_tag de las canciones que lo usan
nombre_tag = {o: texto(inst, t, RBO.lastfm_top_tag) for t, o in inst.subject_objects(RBO.hasTag)}
def fila_tag(s):
    return {'uri': str(s), 'id': local(s).removeprefix('Tag_'),
            'name': nombre_tag.get(s) or local(s).removeprefix('Tag_').replace('_', ' ')}

def fila_etiquetada(g, s):   # Genre / Instrument / Person externos del bonus
    return {'uri': str(s), 'id': local(s).split('_', 1)[-1], 'name': texto(g, s, RDFS.label)}

NODOS = {   # clase RDF -> filas
    RES.Track: [fila_track(s) for s in inst.subjects(RDF.type, RES.Track)],
    RES.Artist: [fila_artist(s) for s in inst.subjects(RDF.type, RES.Artist)],
    RES.Country: [fila_country(s) for s in inst.subjects(RDF.type, RES.Country)],
    RES.Tag: [fila_tag(s) for s in inst.subjects(RDF.type, RES.Tag)],
    RES.Genre: [fila_etiquetada(enr, s) for s in enr.subjects(RDF.type, RES.Genre)],
    RES.Instrument: [fila_etiquetada(enr, s) for s in enr.subjects(RDF.type, RES.Instrument)],
    RES.Person: [fila_etiquetada(enr, s) for s in enr.subjects(RDF.type, RES.Person)],
}
uris_por_clase = {c: {r['uri'] for r in rows} for c, rows in NODOS.items()}

for c, rows in NODOS.items():
    print(f'{local(c):<11} {len(rows):>8,} nodos')
print(f'({time.time() - t0:.1f}s)')

Track        100,000 nodos
Artist        27,378 nodos
Country          138 nodos
Tag              245 nodos
Genre             23 nodos
Instrument         6 nodos
Person             5 nodos
(2.0s)


In [23]:
# --- Relaciones directas (sungBy, hasTag, origin y las del bonus) ---
evidencia = defaultdict(lambda: {'evidence': [], 'sources': []})   # reificación ExtractedFact -> propiedades
for hecho in enr.subjects(RDF.type, RES.ExtractedFact):
    clave = (enr.value(hecho, RDF.subject), enr.value(hecho, RDF.predicate), enr.value(hecho, RDF.object))
    evidencia[clave]['evidence'].append(texto(enr, hecho, RBO.evidenceQuote))
    fuente = enr.value(hecho, RBO.extractedFrom)
    evidencia[clave]['sources'].append(texto(enr, fuente, RBO.sourceUrl))

def uris_de(nombre_clase):
    # URIs cargadas con esa etiqueta (incluye subclases: Artist también es Person)
    return {u for c, us in uris_por_clase.items() if nombre_clase in superclases(c) for u in us}

RELACIONES = []   # (nombre, query, filas)
descartadas = {}
for pred, g in RELACIONES_DIRECTAS:
    dominio, rango = dominio_rango(pred)
    validos_s, validos_o = uris_de(dominio), uris_de(rango)
    filas, n_total = [], 0
    for s, o in g.subject_objects(pred):
        n_total += 1
        if str(s) in validos_s and str(o) in validos_o:
            ev = evidencia.get((s, pred, o), {})
            filas.append({'s': str(s), 'o': str(o), 'evidence': ev.get('evidence'), 'sources': ev.get('sources')})
    descartadas[local(pred)] = n_total - len(filas)
    rel = tipo_relacion(local(pred))
    props = ' SET r.evidence = row.evidence, r.sources = row.sources' if g is enr else ''
    RELACIONES.append((f'{rel}', f'''UNWIND $rows AS row
        MATCH (a:{dominio} {{uri: row.s}}) MATCH (b:{rango} {{uri: row.o}})
        MERGE (a)-[r:{rel}]->(b){props}''', filas))

# --- Similitud Last.fm: nodo en blanco res:Similarity -> relación con propiedad score ---
tracks_uris = uris_por_clase[RES.Track]
similar_rows = []
for s, nodo_sim in inst.subject_objects(RBO.hasSimilarity):
    destino = inst.value(nodo_sim, RBO.similarTrack)
    if str(destino) in tracks_uris:
        similar_rows.append({'s': str(s), 'o': str(destino),
                             'score': float(str(inst.value(nodo_sim, RBO.similarityScore)))})
descartadas['hasSimilarity (canción externa)'] = sim_total - len(similar_rows)
RELACIONES.append(('SIMILAR_TO', '''UNWIND $rows AS row
    MATCH (a:Track {uri: row.s}) MATCH (b:Track {uri: row.o})
    MERGE (a)-[r:SIMILAR_TO]->(b) SET r.score = row.score''', similar_rows))

for nombre, _, filas in RELACIONES:
    print(f'{nombre:<24} {len(filas):>8,} relaciones')
print('Triples no cargables (extremo inexistente en el dataset):', {k: v for k, v in descartadas.items() if v})

SUNG_BY                   100,000 relaciones
HAS_TAG                       475 relaciones
ORIGIN                     22,982 relaciones
ASSOCIATED_WITH_GENRE          11 relaciones
INFLUENCED_BY_GENRE            12 relaciones
PLAYS_INSTRUMENT               17 relaciones
INFLUENCED_BY_ARTIST            0 relaciones
COLLABORATED_WITH               1 relaciones
PRODUCED_BY                     5 relaciones
SIMILAR_TO                  6,380 relaciones
Triples no cargables (extremo inexistente en el dataset): {'hasSimilarity (canción externa)': 54582}


## 7. Importación de instancias → Neo4j

Primero los nodos (con todas las etiquetas de su jerarquía de clases) y luego las relaciones.

In [24]:
t0 = time.time()
for clase, filas in NODOS.items():
    etiquetas = superclases(clase)            # p. ej. ['Track', 'CreativeWork']
    extra = ''.join(f':{e}' for e in etiquetas[1:])
    query = f'UNWIND $rows AS row MERGE (n:{etiquetas[0]} {{uri: row.uri}}) SET n += row'
    query += f' SET n{extra}' if extra else ''
    cargar_en_lotes(f"Nodos :{':'.join(etiquetas)}", query, filas)

for nombre, query, filas in RELACIONES:
    cargar_en_lotes(f'Relaciones :{nombre}', query, filas)
print(f'Carga total: {time.time() - t0:.1f}s')

✓ Nodos :Track:CreativeWork                 100,000  (71.5s)


Transaction failed and will be retried in 1.0690330612278025s ({neo4j_code: Neo.TransientError.Transaction.DeadlockDetected} {message: ForsetiClient[transactionId=375, clientId=2] can't acquire EXCLUSIVE NODE(117442) because it would form this deadlock wait cycle:
NODE(117442)-[EXCLUSIVE_OWNER]->(tx:377)-[WAITING_FOR_EXCLUSIVE]->(NODE(117220))-[EXCLUSIVE_OWNER]->(tx:375)-[WAITING_FOR_EXCLUSIVE]->(NODE(117442))} {gql_status: 50N05} {gql_status_description: error: general processing exception - deadlock detected. Deadlock detected while trying to acquire locks. See log for more details.})


✓ Nodos :Artist:Person                       27,378  (25.8s)
✓ Nodos :Country:GeographicalEntity             138  (0.6s)
✓ Nodos :Tag                                    245  (0.5s)
✓ Nodos :Genre                                   23  (0.7s)
✓ Nodos :Instrument                               6  (0.5s)
✓ Nodos :Person                                   5  (0.5s)
✓ Relaciones :SUNG_BY                       100,000  (77.6s)
✓ Relaciones :HAS_TAG                           475  (0.6s)
✓ Relaciones :ORIGIN                         22,982  (13.6s)
✓ Relaciones :ASSOCIATED_WITH_GENRE              11  (0.6s)
✓ Relaciones :INFLUENCED_BY_GENRE                12  (0.6s)
✓ Relaciones :PLAYS_INSTRUMENT                   17  (0.5s)
✓ Relaciones :INFLUENCED_BY_ARTIST                0  (0.0s)
✓ Relaciones :COLLABORATED_WITH                   1  (0.6s)
✓ Relaciones :PRODUCED_BY                         5  (0.6s)
✓ Relaciones :SIMILAR_TO                      6,380  (4.3s)
Carga total: 199.1s


## 8. Verificación de la carga con Cypher

### 8.1 Conteos: RDF vs Neo4j

Se compara lo esperado (calculado desde los triples RDF) con lo que devuelve Neo4j.

In [25]:
esperado_nodos = {
    'Track': len(NODOS[RES.Track]), 'CreativeWork': len(NODOS[RES.Track]),
    'Artist': len(NODOS[RES.Artist]), 'Person': len(NODOS[RES.Artist]) + len(NODOS[RES.Person]),
    'Country': len(NODOS[RES.Country]), 'GeographicalEntity': len(NODOS[RES.Country]),
    'Tag': len(NODOS[RES.Tag]), 'Genre': len(NODOS[RES.Genre]), 'Instrument': len(NODOS[RES.Instrument]),
    'OntologyClass': len(class_rows), 'OntologyProperty': len(prop_rows),
    'Datatype': len(datatype_rows), 'Restriction': len(restriction_rows),
}
filas = []
for etiqueta, esperado in esperado_nodos.items():
    n = run_cypher(f'MATCH (n:{etiqueta}) RETURN count(n) AS n')[0]['n']
    filas.append({'etiqueta': etiqueta, 'esperado_rdf': esperado, 'neo4j': n, 'ok': n == esperado})
verif_nodos = pd.DataFrame(filas)
display(verif_nodos)

,etiqueta,esperado_rdf,neo4j,ok
0,Track,100000,100000,True
1,CreativeWork,100000,100000,True
2,Artist,27378,27378,True
3,Person,27383,27383,True
4,Country,138,138,True
5,GeographicalEntity,138,138,True
6,Tag,245,245,True
7,Genre,23,23,True
8,Instrument,6,6,True
9,OntologyClass,12,12,True


In [26]:
esperado_rels = {nombre: len(filas) for nombre, _, filas in RELACIONES}
filas = []
for tipo, esperado in esperado_rels.items():
    n = (run_cypher(f'MATCH ()-[r:{tipo}]->() RETURN count(r) AS n')[0]['n'] if esperado
         else run_cypher('MATCH ()-[r]->() WHERE type(r) = $t RETURN count(r) AS n', t=tipo)[0]['n'])
    filas.append({'relación': tipo, 'esperado_rdf': esperado, 'neo4j': n, 'ok': n == esperado})
verif_rels = pd.DataFrame(filas)
display(verif_rels)

assert verif_nodos['ok'].all() and verif_rels['ok'].all(), 'Los conteos no coinciden. Revisa la carga.'
print('✓ Todos los conteos coinciden con el RDF.')

,relación,esperado_rdf,neo4j,ok
0,SUNG_BY,100000,100000,True
1,HAS_TAG,475,475,True
2,ORIGIN,22982,22982,True
3,ASSOCIATED_WITH_GENRE,11,11,True
4,INFLUENCED_BY_GENRE,12,12,True
5,PLAYS_INSTRUMENT,17,17,True
6,INFLUENCED_BY_ARTIST,0,0,True
7,COLLABORATED_WITH,1,1,True
8,PRODUCED_BY,5,5,True
9,SIMILAR_TO,6380,6380,True


✓ Todos los conteos coinciden con el RDF.


In [27]:
# Vista global: etiquetas y tipos de relación presentes en la base (incluye lo que hubiera de otros proyectos)
display(pd.DataFrame(run_cypher('''
MATCH (n) UNWIND labels(n) AS etiqueta
RETURN etiqueta, count(*) AS nodos ORDER BY nodos DESC''')))
display(pd.DataFrame(run_cypher('''
MATCH (a)-[r]->(b)
WITH [l IN labels(a) WHERE NOT l IN $super][0] AS origen, type(r) AS relacion,
     [l IN labels(b) WHERE NOT l IN $super][0] AS destino
RETURN origen, relacion, destino, count(*) AS total
ORDER BY total DESC''', super=['CreativeWork', 'GeographicalEntity'])))

,etiqueta,nodos
0,Track,100000
1,CreativeWork,100000
2,Person,27383
3,Artist,27378
4,Tag,245
5,Country,138
6,GeographicalEntity,138
7,OntologyProperty,28
8,Genre,23
9,OntologyClass,12


,origen,relacion,destino,total
0,Track,SUNG_BY,Artist,100000
1,Artist,ORIGIN,Country,22982
2,Track,SIMILAR_TO,Track,6380
3,Track,HAS_TAG,Tag,475
4,OntologyProperty,DOMAIN,OntologyClass,25
5,Artist,PLAYS_INSTRUMENT,Instrument,17
6,OntologyClass,DISJOINT_WITH,OntologyClass,15
7,OntologyProperty,RANGE,Datatype,14
8,OntologyProperty,RANGE,OntologyClass,12
9,Artist,INFLUENCED_BY_GENRE,Genre,12


### 8.2 Integridad respecto a la ontología

La ontología exige que todo `Track` tenga **al menos un** `Artist` (`owl:minQualifiedCardinality 1` sobre `sungBy`)
y **exactamente un** título (`owl:cardinality 1`). Se leen las restricciones desde el meta-grafo y se comprueban sobre los datos.

In [28]:
display(pd.DataFrame(run_cypher('''
MATCH (c:OntologyClass)-[:SUBCLASS_OF]->(r:Restriction)-[:ON_PROPERTY]->(p:OntologyProperty)
OPTIONAL MATCH (r)-[:ON_CLASS]->(oc:OntologyClass)
RETURN c.name AS clase, p.name AS propiedad, r.kind AS restriccion, r.value AS valor, oc.name AS sobre_clase''')))

integridad = run_cypher('''
CALL () { MATCH (t:Track) WHERE NOT (t)-[:SUNG_BY]->(:Artist) RETURN count(t) AS tracks_sin_artista }
CALL () { MATCH (t:Track)-[:SUNG_BY]->(a) WITH t, count(a) AS n WHERE n > 1 RETURN count(t) AS tracks_con_varios_artistas }
CALL () { MATCH (t:Track) WHERE t.title IS NULL RETURN count(t) AS tracks_sin_titulo }
CALL () { MATCH (a:Artist) WHERE NOT ()-[:SUNG_BY]->(a) RETURN count(a) AS artistas_sin_canciones }
CALL () { MATCH (a:Artist) WHERE NOT (a)-[:ORIGIN]->(:Country) RETURN count(a) AS artistas_sin_pais }
CALL () { MATCH (t:Track) WHERE t.year IS NULL RETURN count(t) AS tracks_sin_anio }
RETURN *''')
display(pd.DataFrame(integridad))

,clase,propiedad,restriccion,valor,sobre_clase
0,Track,sungBy,minQualifiedCardinality,1,Artist
1,Track,sungBy,allValuesFrom,Artist,Artist
2,Artist,sing,allValuesFrom,Track,Track
3,Track,title,cardinality,1,None


,artistas_sin_canciones,artistas_sin_pais,tracks_con_varios_artistas,tracks_sin_anio,tracks_sin_artista,tracks_sin_titulo
0,0,4396,0,45346,0,3


> **Nota.** Unos pocos tracks no tienen título porque tampoco lo tienen en el RDF de la Etapa 2 (incumplen la
> restricción `cardinality 1`). Es un hallazgo de calidad de datos que la verificación hace visible; no un error de carga.

### 8.3 La ontología consultada desde Neo4j

In [29]:
print('Jerarquía de clases')
display(pd.DataFrame(run_cypher('''
MATCH (c:OntologyClass)-[r:SUBCLASS_OF]->(p:OntologyClass)
RETURN c.name AS subclase, p.name AS superclase, r.via AS axioma ORDER BY superclase, subclase''')))

print('Propiedades: dominio, rango y representación en Neo4j')
display(pd.DataFrame(run_cypher('''
MATCH (p:OntologyProperty)
OPTIONAL MATCH (p)-[:DOMAIN]->(d)
OPTIONAL MATCH (p)-[:RANGE]->(r)
OPTIONAL MATCH (p)-[:SUBPROPERTY_OF]->(sp)
OPTIONAL MATCH (p)-[:INVERSE_OF]-(inv)
RETURN p.name AS propiedad, p.kind AS tipo, d.name AS dominio, r.name AS rango,
       sp.name AS subpropiedad_de, inv.name AS inversa, p.chain AS cadena, p.neo4j_mapping AS en_neo4j
ORDER BY tipo, propiedad''')))

Jerarquía de clases


,subclase,superclase,axioma
0,Track,CreativeWork,rdf:subClassOf
1,Country,GeographicalEntity,rdf:subClassOf
2,Artist,Person,rdf:subClassOf


Propiedades: dominio, rango y representación en Neo4j


,propiedad,tipo,dominio,rango,subpropiedad_de,inversa,cadena,en_neo4j
0,area,DatatypeProperty,Country,xsd:string,None,None,None,Country.areas
1,artist_type,DatatypeProperty,Artist,xsd:string,None,None,None,Artist.artist_type
2,duration,DatatypeProperty,Track,xsd:double,None,None,None,Track.duration
3,evidenceQuote,DatatypeProperty,ExtractedFact,xsd:string,None,None,None,None
4,gender,DatatypeProperty,Artist,xsd:string,None,None,None,Artist.gender
5,key,DatatypeProperty,Track,xsd:string,None,None,None,Track.key
6,keyScale,DatatypeProperty,Track,xsd:string,None,None,None,Track.keyScale
7,lastfm_top_tag,DatatypeProperty,Track,xsd:string,None,None,None,Track.lastfm_top_tag
8,lastfm_top_tag_score,DatatypeProperty,Track,xsd:double,None,None,None,Track.lastfm_top_tag_score
9,released,DatatypeProperty,Track,xsd:string,None,None,None,Track.released


### 8.4 Recuperación de las entidades y relaciones principales

In [30]:
# Una canción "rica" para inspeccionar: con tag, con similares y con país de origen
ejemplo = run_cypher('''
MATCH (t:Track)-[:HAS_TAG]->(:Tag)
WHERE (t)-[:SIMILAR_TO]->() AND (t)-[:SUNG_BY]->(:Artist)-[:ORIGIN]->(:Country)
RETURN t.id AS id ORDER BY t.id LIMIT 1''')
TRACK_EJEMPLO = ejemplo[0]['id'] if ejemplo else run_cypher('MATCH (t:Track) RETURN t.id AS id LIMIT 1')[0]['id']

print('A. Ficha de una canción: Track → Artist → Country, Track → Tag')
display(pd.DataFrame(run_cypher('''
MATCH (t:Track {id: $id})-[:SUNG_BY]->(a:Artist)
OPTIONAL MATCH (a)-[:ORIGIN]->(c:Country)
OPTIONAL MATCH (t)-[:HAS_TAG]->(tag:Tag)
RETURN t.id AS id, t.title AS titulo, t.released AS album, t.year AS anio, round(t.duration, 1) AS duracion_s,
       t.key + ' ' + t.keyScale AS tonalidad, a.name AS artista, a.artist_type AS tipo_artista,
       c.id AS pais, tag.name AS tag, labels(t) AS etiquetas''', id=TRACK_EJEMPLO)).T)

A. Ficha de una canción: Track → Artist → Country, Track → Tag


,0
id,TRAABJV128F1460C49
titulo,Tonight Will Be Alright
album,Dancing On The Ceiling
anio,1986
duracion_s,307.4
tonalidad,None
artista,Lionel Richie
tipo_artista,Person
pais,US
tag,slowhand


In [31]:
print('B. Canciones más similares a la canción de ejemplo (SIMILAR_TO con score)')
display(pd.DataFrame(run_cypher('''
MATCH (t:Track {id: $id})-[s:SIMILAR_TO]->(o:Track)-[:SUNG_BY]->(a:Artist)
RETURN o.id AS id, o.title AS titulo, a.name AS artista, round(s.score, 4) AS score
ORDER BY s.score DESC LIMIT 10''', id=TRACK_EJEMPLO)))

B. Canciones más similares a la canción de ejemplo (SIMILAR_TO con score)


,id,titulo,artista,score
0,TRKYUDS128F931BF1D,Forever Came Today,Jackson 5,0.0019
1,TRVGBDS128F9364C50,Kyrie,Mr Mister,0.0014
2,TREPDJZ128F1495CA5,The Fool On The Hill,Four Tops,0.0014
3,TRIKKYP128F425D8EC,If You Don't Want My Love,Richard Marx,0.0012
4,TRTZBRA128F92E2368,Riptide,Robert Palmer,0.0011
5,TRPFBNL128F42A39C1,Tell It To My Heart,Taylor Dayne,0.0007


In [32]:
print('C. Artistas con más canciones en el grafo')
top_artistas = pd.DataFrame(run_cypher('''
MATCH (t:Track)-[:SUNG_BY]->(a:Artist)
OPTIONAL MATCH (a)-[:ORIGIN]->(c:Country)
RETURN a.name AS artista, c.id AS pais, count(t) AS canciones
ORDER BY canciones DESC, artista LIMIT 10'''))
display(top_artistas)

ARTISTA_EJEMPLO = top_artistas.loc[0, 'artista']
print(f'D. Canciones de un artista concreto: {ARTISTA_EJEMPLO}')
display(pd.DataFrame(run_cypher('''
MATCH (t:Track)-[:SUNG_BY]->(a:Artist {name: $artista})
RETURN t.title AS titulo, t.released AS album, t.year AS anio
ORDER BY anio DESC, titulo LIMIT 10''', artista=ARTISTA_EJEMPLO)))

C. Artistas con más canciones en el grafo


,artista,pais,canciones
0,Ike & Tina Turner,US,29
1,Billy Bragg,GB,28
2,Sonny Rollins,US,27
3,Astrud Gilberto,BR,25
4,Gary Numan,GB,25
5,Lyala,US,25
6,Nina Simone,US,25
7,Salt-N-Pepa,US,25
8,Bob Dylan,US,24
9,Leonard Cohen,CA,24


D. Canciones de un artista concreto: Ike & Tina Turner


,titulo,album,anio
0,Be My Love,Best of,NaN
1,Desire,Don't Play Me Cheap/It's Gonna Work Out Fine,NaN
2,Fool In Love - Re-Recording,Ike and Tina Turner Selected Favorites,NaN
3,Gonna Find me A Substitute,The Archive Series Vol. 3 - Movin',NaN
4,I Know (You Don't Want Me No More) (Live Version),Live! The Ike & Tina Turner Show,NaN
5,Nudden,Best of,NaN
6,Something's Got A Hold On Me (Live Version),Live! The Ike & Tina Turner Show,NaN
7,Still Keep Missing You,The Archive Series Vol. 6 - Rollin',NaN
8,Three O'clock In The Morning Blues,Sing The Blues,NaN
9,The Locomotion,The Ike & Tina Turner Archive Series : Hits & Classics Vol.1 & 2,2010.0


In [33]:
print('E. Países con más artistas (y muestra de sus áreas)')
display(pd.DataFrame(run_cypher('''
MATCH (a:Artist)-[:ORIGIN]->(c:Country)
RETURN c.id AS pais, count(a) AS artistas, c.n_areas AS n_areas, c.areas[0..5] AS ejemplo_areas
ORDER BY artistas DESC LIMIT 10''')))

print('F. Tags Last.fm más frecuentes')
display(pd.DataFrame(run_cypher('''
MATCH (t:Track)-[:HAS_TAG]->(tag:Tag)
RETURN tag.name AS tag, count(t) AS canciones
ORDER BY canciones DESC, tag LIMIT 10''')))

E. Países con más artistas (y muestra de sus áreas)


,pais,artistas,n_areas,ejemplo_areas
0,US,9840,198,"[Alabama, Alameda, Albany, Albuquerque, Allentown]"
1,GB,3488,72,"[Abingdon-on-Thames, Aylesbury, Barking, Bath, Belfast]"
2,DE,1403,33,"[Baden-Württemberg, Berlin, Bochum, Bottrop, Darmstadt]"
3,FR,882,20,"[Bayeux, Bordeaux, Boulogne-sur-Mer, Bretagne, Chilly-Mazarin]"
4,CA,762,20,"[Calgary, Canada, Gatineau, Guelph, Hamilton]"
5,SE,684,16,"[Borlänge, Gothenburg, Luleå, Lund, Lysekil Municipality]"
6,IT,550,7,"[Brescia, Italy, Milano, Ravenna, Roma]"
7,ES,530,18,"[Almería, Asturias, Barcelona, Catalunya, Ferrol]"
8,AU,398,11,"[Adelaide, Australia, Brisbane, Geelong, Melbourne]"
9,NL,355,9,"[Amsterdam, Den Haag, Dordrecht, Haarlem, Limburg]"


F. Tags Last.fm más frecuentes


,tag,canciones
0,rock,27
1,blues,20
2,country,15
3,pop,15
4,Hip-Hop,12
5,punk,12
6,jazz,11
7,classic rock,10
8,soul,10
9,electronic,9


In [34]:
print('G. Cadena de propiedades rbo:sungInCountry ≡ sungBy ∘ origin, evaluada con un camino Cypher')
display(pd.DataFrame(run_cypher('''
MATCH (p:OntologyProperty {name: 'sungInCountry'})-[c:CHAIN_STEP]->(paso)
WITH p, c, paso ORDER BY c.order
RETURN p.name AS propiedad, collect(paso.name) AS pasos''')))
display(pd.DataFrame(run_cypher('''
MATCH (t:Track)-[:SUNG_BY]->(:Artist)-[:ORIGIN]->(c:Country)
RETURN c.id AS pais, count(t) AS canciones_sung_in_country
ORDER BY canciones_sung_in_country DESC LIMIT 10''')))

G. Cadena de propiedades rbo:sungInCountry ≡ sungBy ∘ origin, evaluada con un camino Cypher


,propiedad,pasos
0,sungInCountry,"[sungBy, origin]"


,pais,canciones_sung_in_country
0,US,40293
1,GB,13815
2,DE,4841
3,FR,3230
4,CA,2685
5,SE,2397
6,ES,2182
7,IT,1897
8,JM,1473
9,AU,1315


In [35]:
print('H. Jerarquía materializada: una canción es también CreativeWork; un artista es también Person')
display(pd.DataFrame(run_cypher('''
MATCH (n) WHERE n:CreativeWork OR n:Person OR n:GeographicalEntity
UNWIND labels(n) AS etiqueta
WITH etiqueta, count(*) AS nodos WHERE etiqueta IN ['CreativeWork','Track','Person','Artist','GeographicalEntity','Country']
RETURN etiqueta, nodos ORDER BY etiqueta''')))

H. Jerarquía materializada: una canción es también CreativeWork; un artista es también Person


,etiqueta,nodos
0,Artist,27378
1,Country,138
2,CreativeWork,100000
3,GeographicalEntity,138
4,Person,27383
5,Track,100000


In [36]:
if INCLUIR_ENRIQUECIMIENTO:
    print('I. Bonus: conocimiento extraído de textos (géneros, instrumentos, personas) con su evidencia')
    display(pd.DataFrame(run_cypher('''
    MATCH (a:Artist)-[r]->(x) WHERE type(r) IN $tipos
    RETURN a.name AS artista, type(r) AS relacion, x.name AS objeto, r.evidence[0] AS evidencia, r.sources[0] AS fuente
    ORDER BY artista, relacion LIMIT 15''',
        tipos=[tipo_relacion(local(p)) for p, g in RELACIONES_DIRECTAS if g is enr])))

I. Bonus: conocimiento extraído de textos (géneros, instrumentos, personas) con su evidencia


,artista,relacion,objeto,evidencia,fuente
0,!!!,PLAYS_INSTRUMENT,guitar,Mario Andreoni (guitar),https://en.wikipedia.org/wiki/!!!
1,!!!,PLAYS_INSTRUMENT,drums,Chris Egan (drums),https://en.wikipedia.org/wiki/!!!
2,!!!,PLAYS_INSTRUMENT,bass,Rafael Cohen (bass/various electronic devices),https://en.wikipedia.org/wiki/!!!
3,#Poundsign#,ASSOCIATED_WITH_GENRE,indie pop,The Aislers Set is an American indie pop band,https://en.wikipedia.org/wiki/The_Aislers_Set
4,#Poundsign#,INFLUENCED_BY_GENRE,C86-style British indie pop,The Aislers Set's music is often interpreted as adjacent to C86-style British indie pop,https://en.wikipedia.org/wiki/The_Aislers_Set
5,#Poundsign#,PLAYS_INSTRUMENT,organ,Jen Cohen (organ),https://en.wikipedia.org/wiki/The_Aislers_Set
6,#Poundsign#,PLAYS_INSTRUMENT,guitar,"AV Linton (guitar, vocals, multi-instrumentalist)",https://en.wikipedia.org/wiki/The_Aislers_Set
7,#Poundsign#,PLAYS_INSTRUMENT,bass,"Alicia Vanden Heuvel, (bass, vocals, piano, multi-instrumentalist)",https://en.wikipedia.org/wiki/The_Aislers_Set
8,#Poundsign#,PLAYS_INSTRUMENT,drums,Yoshi Nakamoto (drums),https://en.wikipedia.org/wiki/The_Aislers_Set
9,& And Oceans,ASSOCIATED_WITH_GENRE,metal,"...And Oceans, previously Havoc Unit and Festerday, is a Finnish metal band.",https://en.wikipedia.org/wiki/...And_Oceans


### 8.5 Consulta preparatoria para la recomendación

Vecindario de una canción en el grafo: canciones del mismo artista, del mismo país y similares según Last.fm.
Estas son las señales que el sistema de recomendación (siguiente punto) podrá combinar.

In [37]:
display(pd.DataFrame(run_cypher('''
MATCH (t:Track {id: $id})-[:SUNG_BY]->(a:Artist)
CALL (t, a) { MATCH (a)<-[:SUNG_BY]-(o:Track) WHERE o <> t RETURN count(o) AS mismo_artista }
CALL (t, a) { MATCH (a)-[:ORIGIN]->(:Country)<-[:ORIGIN]-(:Artist)<-[:SUNG_BY]-(o:Track) RETURN count(o) AS mismo_pais }
CALL (t)    { MATCH (t)-[:SIMILAR_TO]-(o:Track) RETURN count(DISTINCT o) AS similares_lastfm }
CALL (t)    { MATCH (t)-[:HAS_TAG]->(:Tag)<-[:HAS_TAG]-(o:Track) WHERE o <> t RETURN count(o) AS mismo_tag }
RETURN t.title AS cancion, a.name AS artista, mismo_artista, mismo_pais, similares_lastfm, mismo_tag''',
    id=TRACK_EJEMPLO)))

,cancion,artista,mismo_artista,mismo_pais,similares_lastfm,mismo_tag
0,Tonight Will Be Alright,Lionel Richie,14,40278,6,0


## 9. Resumen

In [38]:
resumen = run_cypher('''
MATCH (n) WHERE any(l IN labels(n) WHERE l IN $labels)
WITH count(n) AS nodos
MATCH (a)-[r]->(b) WHERE any(l IN labels(a) WHERE l IN $labels)
RETURN nodos, count(r) AS relaciones''', labels=ETIQUETAS_PROYECTO)[0]
print(f"Grafo del proyecto en Neo4j: {resumen['nodos']:,} nodos y {resumen['relaciones']:,} relaciones.")
print('Verificación de conteos:', 'OK' if verif_nodos['ok'].all() and verif_rels['ok'].all() else 'CON DIFERENCIAS')

Grafo del proyecto en Neo4j: 127,843 nodos y 129,968 relaciones.
Verificación de conteos: OK


In [39]:
driver.close()